# Google Trends 1-year regression: search terms versus GP admissions

This notebook uses the one-year Google Trends search-term files in:

```text
Google_trends_v2/1y_data
```

The files are expected to have filenames beginning with `time_series_GB`. The predictive variable from each file is the **second source column**, regardless of its name. That selected second column is converted to a canonical `count` field inside the analysis code.

## Model

The default model treats the one-year Google Trends files as separate predictive variables:

```text
z(GP admissions)_t = beta_0 + beta_1 z(term 1)_t + beta_2 z(term 1)_{t-1} + beta_3 z(term 2)_t + ... + error_t
```

Set `AGGREGATE_TERMS = True` if you instead want a single predictor formed by summing all second-column values across the selected search-term files.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from wastewater.io import find_repo_root
from wastewater.search_terms import (
    find_search_term_files,
    build_search_term_catalogue,
    search_file_to_long,
    build_search_term_regression_frame,
    search_term_lagged_predictor_columns,
    fit_search_term_ols,
)
from wastewater.ukhsa import build_ukhsa_series_catalogue

ROOT = find_repo_root(ROOT)
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

SEARCH_TERMS_DIR = Path('Google_trends_v2') / '1y_data'
ROOT

## 1. Find one-year Google Trends files

This scans only `Google_trends_v2/1y_data` for files beginning with `time_series_GB`.

In [ ]:
search_files = find_search_term_files(ROOT, search_dir=SEARCH_TERMS_DIR)
display(search_files.sort_values('path'))
print(f'Found {len(search_files)} one-year Google Trends files')

## 2. Inspect selected predictor columns

The important check is the `predictor_column` column below. It should be the second source column in each one-year Google Trends file. Python uses zero-based indexing, so `VALUE_COLUMN_INDEX = 1` means the second column.

In [ ]:
VALUE_COLUMN_INDEX = 1  # second source column

search_catalogue = build_search_term_catalogue(
    ROOT,
    search_dir=SEARCH_TERMS_DIR,
    value_column_index=VALUE_COLUMN_INDEX,
)
display(search_catalogue[['path', 'filename', 'date_column', 'predictor_column_index', 'predictor_column', 'usable_predictor_values', 'status', 'error']].sort_values('path'))

# Full column list for debugging.
display(search_catalogue[['path', 'columns']])

## 3. Choose GP admissions outcome files

The outcome is taken from the UKHSA chart files classified as GP/admissions. If inference is wrong, manually set `OUTCOME_FILES` using paths from the catalogue.

In [ ]:
ukhsa_catalogue = build_ukhsa_series_catalogue(ROOT)
display(ukhsa_catalogue[['path', 'filename', 'series_type', 'date_column', 'value_column']].sort_values(['series_type', 'path']))

SEARCH_FILES = search_catalogue.loc[search_catalogue['status'] == 'ok', 'path'].tolist()
OUTCOME_FILES = ukhsa_catalogue.loc[ukhsa_catalogue['series_type'] == 'gp_admissions', 'path'].tolist()

# Manual override examples:
# SEARCH_FILES = ['Google_trends_v2/1y_data/time_series_GB_...csv']
# OUTCOME_FILES = ['data/raw/ukhsa-chart-...gp-admissions...csv']

print('One-year Google Trends predictor files:')
for path in SEARCH_FILES:
    print(' -', path)

print('\nGP admissions outcome files:')
for path in OUTCOME_FILES:
    print(' -', path)

## 4. Preview search-term second-column series

In [ ]:
for path in SEARCH_FILES[:5]:
    print('\nSearch file:', path)
    display(search_file_to_long(ROOT, path, value_column_index=VALUE_COLUMN_INDEX).head())

## 5. Build regression frame

Use weekly aggregation first. Switch `FREQ = 'M'` if the search-term and GP-admission series are monthly. By default, each one-year Google Trends file is treated as a separate predictive variable.

In [ ]:
FREQ = 'W'
LAGS = [0, 1, 2, 3, 4]
AGGREGATE_TERMS = False

regression_frame = build_search_term_regression_frame(
    ROOT,
    search_files=SEARCH_FILES,
    outcome_files=OUTCOME_FILES,
    search_dir=SEARCH_TERMS_DIR,
    freq=FREQ,
    lags=LAGS,
    aggregate_terms=AGGREGATE_TERMS,
    value_column_index=VALUE_COLUMN_INDEX,
)

display(regression_frame)

out = PROCESSED / 'google_trends_1y_gp_admissions_regression_frame.csv'
regression_frame.to_csv(out, index=False)
print(f'Saved {len(regression_frame):,} rows to {out.relative_to(ROOT)}')

## 6. Plot aligned series

For visualisation, this computes the mean z-score across the unlagged Google Trends predictors and plots it against GP admissions. The regression itself uses the individual lagged predictor columns.

In [ ]:
unlagged_search_cols = [
    col for col in regression_frame.columns
    if col.startswith('z_') and '_lag' not in col and col != 'z_gp_admissions'
]

plot_df = regression_frame.copy()
if AGGREGATE_TERMS:
    plot_df['z_mean_google_trends'] = plot_df['z_search_count']
else:
    plot_df['z_mean_google_trends'] = plot_df[unlagged_search_cols].mean(axis=1)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(plot_df['period'], plot_df['z_mean_google_trends'], label='Mean Google Trends predictor, z-scored')
ax.plot(plot_df['period'], plot_df['z_gp_admissions'], label='GP admissions, z-scored')
ax.set_title('Google Trends 1-year predictors and GP admissions')
ax.set_xlabel('Period')
ax.set_ylabel('z-score')
ax.legend()
fig.autofmt_xdate()
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(plot_df['z_mean_google_trends'], plot_df['z_gp_admissions'])
ax.set_xlabel('Mean Google Trends predictor, z-scored')
ax.set_ylabel('GP admissions, z-scored')
ax.set_title('Contemporaneous association')
plt.show()

## 7. Fit lagged regression

For one-year data, many predictors plus month dummies can overfit quickly. The default below uses the individual Google Trends predictors with HAC standard errors and no seasonal dummies. Set `SEASONAL_CONTROLS = True` only if the model has enough observations relative to predictors.

In [ ]:
SEASONAL_CONTROLS = False

if AGGREGATE_TERMS:
    predictor_cols = [f'z_search_count_lag{lag}' for lag in LAGS]
else:
    predictor_cols = search_term_lagged_predictor_columns(regression_frame, lags=LAGS)

print(f'Using {len(predictor_cols)} lagged Google Trends predictor columns')
print(predictor_cols)

result = fit_search_term_ols(
    regression_frame,
    lags=LAGS,
    predictor_columns=predictor_cols,
    seasonal_controls=SEASONAL_CONTROLS,
)
print(result.summary())

## 8. Observed versus fitted

In [ ]:
model_rows = regression_frame.dropna(subset=['z_gp_admissions', *predictor_cols]).copy()
model_rows['prediction'] = result.predict()
model_rows['residual'] = model_rows['z_gp_admissions'] - model_rows['prediction']

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(model_rows['period'], model_rows['z_gp_admissions'], label='Observed GP admissions, z')
ax.plot(model_rows['period'], model_rows['prediction'], label='Predicted GP admissions, z')
ax.set_title('Observed versus fitted GP admissions from Google Trends 1-year predictors')
ax.legend()
fig.autofmt_xdate()
plt.show()

model_rows.to_csv(PROCESSED / 'google_trends_1y_gp_admissions_model_rows.csv', index=False)
model_rows[['period', 'z_gp_admissions', 'prediction', 'residual']].head()

## 9. Next refinements

- Reduce `LAGS` if there are too many predictors relative to observations.
- Compare the individual-term model with `AGGREGATE_TERMS = True`.
- Try regularised models such as ridge or lasso if the one-year predictor set is large.
- Compare weekly and monthly aggregation.
- Add UKHSA NHS 111 call series and wastewater predictors after the Google Trends baseline is stable.
- Use train/test splits or rolling-origin validation before treating this as predictive.